# Module: SQL and SQLite Assignments
## Lesson: Working with SQLite using the `sqlite3` module
### Assignment 1: Connect to a Database

Import the `sqlite3` module and connect to a database file named `company.db`. Print the connection object, then confirm (using `os.path.exists`) that the `.db` file was created in your current folder.

In [40]:
import sqlite3

### Assignment 2: Create a Cursor

Create a cursor object from the connection. In a markdown cell or comment, explain in one or two sentences why a cursor is needed to run queries.

In [41]:
conn = sqlite3.connect("company.db")
cursor = conn.cursor() 

### Assignment 3: Create a Table

Using `cursor.execute`, create a table named `employees` **only if it does not already exist**, with the columns:

- `id` — `INTEGER`, primary key
- `name` — `TEXT`, cannot be null
- `age` — `INTEGER`
- `department` — `TEXT`

Commit the change. Run the cell twice to verify that `IF NOT EXISTS` prevents an error.

In [42]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees(
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        age INTEGER,
        department TEXT
    )
""")

conn.commit()

In [43]:
cursor.execute("""
    SELECT * FROM employees
""")

### Assignment 4: Insert Records

Insert at least 5 employees into the `employees` table, one `INSERT` statement per record. Use at least 3 different departments. Remember to commit.

In [44]:
cursor.execute("""
    INSERT INTO employees(name, age, department)
    VALUES
        ("Lin", 23, "engineering"),
        ("Ana", 31, "sales"),
        ("Raj", 28, "marketing"),
        ("Mei", 35, "engineering"),
        ("Tom", 26, "hr")
""")

conn.commit()

OperationalError: database is locked

### Assignment 5: Query All Records

Run `SELECT * FROM employees`, fetch every row with `fetchall()`, and print each row on its own line. What Python type is each row?

In [ ]:
cursor.execute("SELECT * FROM employees")
rows = cursor.fetchall()
rows

### Assignment 6: Fetch a Single Record

Use `fetchone()` to get only the first employee whose department is `'Engineering'` (or any department you inserted). Print the result. What does `fetchone()` return when no row matches?

In [ ]:
cursor.execute("SELECT * FROM employees WHERE department = 'engineering'")
row = cursor.fetchone()
row

### Assignment 7: Parameterized Queries
Write a function `get_employees_by_department(cursor, department)` that returns all employees in the given department. Use a `?` placeholder instead of string formatting (f-strings) to pass the department. In a comment, explain why placeholders are safer.

In [ ]:
def get_employees_by_department(cursor: sqlite3.Cursor, department: str) -> list:
    cursor.execute("SELECT * FROM employees WHERE department = ?", (department,))
    return cursor.fetchall()

### Assignment 8: Update a Record

Increase the age of one employee by 2 using an `UPDATE ... SET ... WHERE` statement. Commit, then query the table again to confirm the change.

In [ ]:
cursor.execute("UPDATE employees SET age = age + 2 WHERE id = 1")
conn.commit()

In [ ]:
cursor.execute("SELECT * FROM employees")

for x in cursor.fetchall():
    print(x)

### Assignment 9: Delete a Record

Delete one employee by name using `DELETE FROM ... WHERE`. Commit, then print all remaining rows and the total number of employees (use `SELECT COUNT(*)`).

In [ ]:
cursor.execute("DELETE FROM employees WHERE id = 5")
conn.commit()

cursor.execute("SELECT * FROM employees")

for row in cursor.fetchall():
    print(row)

### Assignment 10: Forgetting to Commit

Open a **second** connection to `company.db`. In the first connection, insert a new employee but **do not** commit. Query the table from the second connection — is the new employee visible? Now commit from the first connection and query again. Write down what you observed.

In [ ]:
conn_2 = sqlite3.connect("company.db")
cursor_2 = conn_2.cursor()

cursor.execute("""
    INSERT INTO employees(name, age, department)
    VALUES
        ('Tom', 26, 'hr')
""")

cursor_2.execute("""
    SELECT * FROM employees WHERE name = 'Tom'
""")

print(cursor_2.fetchone())

In [ ]:
cursor.execute("""
    INSERT INTO employees(name, age, department)
    VALUES
        ('Tom', 26, 'hr')
""")

conn.commit()

cursor_2.execute("""
    SELECT * FROM employees WHERE name = 'Tom'
""")

print(cursor_2.fetchone())

### Assignment 11: Bulk Insert with `executemany`

Connect to a new database `sales_data.db` and create a table `sales` with columns `id` (`INTEGER` primary key), `date` (`TEXT`, not null), `product` (`TEXT`, not null), `sales` (`INTEGER`) and `region` (`TEXT`).

Create a list of at least 8 tuples like `('2026-01-01', 'Laptop', 100, 'North')` and insert them all at once with `cursor.executemany` using `?` placeholders. Commit and print every row.

In [ ]:
conn = sqlite3.connect("sales_data.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS sales(
        id INTEGER PRIMARY KEY, 
        date TEXT NOT NULL,
        product TEXT NOT NULL,
        sales INTEGER,
        region TEXT
    ) 
""")

conn.commit()

sales = [
    ("2026-01-01", "Laptop", 100, "North"),
    ("2026-09-01", "Bag", 10000, "South"),
    ("2026-06-01", "Tea", 200, "North"),
    ("2026-01-01", "Sugar", 34, "North"),
    ("2026-02-01", "Water Bottle", 5, "East"),
    ("2026-05-01", "Chips", 2000, "West"),
    ("2026-11-01", "Biscuit", 113, "North"),
    ("2026-07-01", "Flour", 100, "East"),
]
cursor.executemany(
    """
        INSERT INTO sales(date, product, sales, region)
        VALUES (?, ?, ?, ?)
    """,
    sales,
)

conn.commit()

cursor.execute("SELECT * FROM sales")

for row in cursor.fetchall():
    print(row)

### Assignment 12: Analyze the Sales Data

Using SQL queries on the `sales` table, find and print:

1. Total sales per region (`GROUP BY`)
2. The product with the highest single sale
3. All sales above 150, ordered from highest to lowest
4. The average sales value for each product

In [ ]:
cursor.execute("""
    SELECT region, SUM(sales) AS total_sales 
    FROM sales 
    GROUP BY region
    ORDER BY total_sales DESC
""")

for rows in cursor.fetchall():
    print(rows)

In [ ]:
cursor.execute("""
    SELECT region, SUM(sales) AS total_sales 
    FROM sales 
    GROUP BY region
    ORDER BY total_sales DESC
""")

print(cursor.fetchone())

In [ ]:
cursor.execute("SELECT * FROM sales WHERE sales > 150")

for rows in cursor.fetchall():
    print(rows)

In [ ]:
cursor.execute("SELECT product, AVG(sales) AS avg_sales FROM sales GROUP BY product ORDER BY avg_sales DESC")

for rows in cursor.fetchall():
    print(rows)

### Assignment 13: Close the Connection

Close the `sales_data.db` connection with `connection.close()`. Then try to run a query on it and catch the resulting exception with `try`/`except`. Print the exception type and message.

In [ ]:
conn.close()

In [ ]:
try:
    cursor.execute("SELECT product, AVG(sales) AS avg_sales FROM sales GROUP BY product ORDER BY avg_sales DESC")

    for rows in cursor.fetchall():
        print(rows)
except sqlite3.ProgrammingError as ex:
    print(ex)

### Assignment 14: Mini Project: Student Records Manager

Build a small set of functions on top of a `students.db` database with a `students` table (`id`, `name`, `grade`, `marks`):

- `add_student(name, grade, marks)`
- `add_students(list_of_tuples)` — uses `executemany`
- `get_all_students()`
- `update_marks(name, new_marks)`
- `remove_student(name)`
- `top_students(n)` — returns the `n` students with the highest marks

Every function that changes data must commit, and all queries must use `?` placeholders. Demonstrate each function, then close the connection at the end.

In [ ]:
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS students(
        id INTEGER PRIMARY KEY,
        name TEXT,
        grade FLOAT,
        marks FLOAT
    )
""")

conn.commit()

In [ ]:
from typing import Any

def execute(statement: str, data: Any = None, is_return: bool = False) -> None | list:
    if data is None:
        cursor.execute(statement)
    else:
        cursor.execute(statement, data)

    if is_return: 
        return cursor.fetchall()
    
    conn.commit()

In [ ]:
def add_students(students: list) -> None:
    execute(
        """
            INSERT INTO students(name, grade, marks)
            VALUES
                (?, ?, ?)
        """,
        students,
    )

In [ ]:
def add_student( name: str, grade: float, marks: float, ) -> None:
    add_students([(name, grade, marks)])

In [ ]:
def get_all_students() -> list:
    return execute("SELECT * FROM students", is_return=True)

In [ ]:
def update_marks(name: str, new_marks: float):
    execute("UPDATE students SET marks = ? WHERE name = ?", (new_marks, name))

In [ ]:
def remove_student(name: str) -> None:
    execute("DELETE FROM students WHERE name = ?", name)

In [ ]:
def top_students(n: float) -> list:
    cursor.execute("SELECT * FROM students WHERE marks > n")
    return cursor.fetchall()